# Train Qwen Small for Claim Extraction

This notebook fine-tunes a small Qwen model on `data/claims.jsonl` to produce outputs in this strict format:

```json
{"input": "...", "claims": ["..."]}
```

The training approach uses TRL `SFTTrainer` + PEFT LoRA.

In [ ]:
import os
import json
import random
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cpu
CUDA available: False


In [ ]:
# Resolve data path robustly whether notebook runs from repo root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'claims.jsonl').exists() and (ROOT.parent / 'data' / 'claims.jsonl').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'claims.jsonl'
assert DATA_PATH.exists(), f'Cannot find dataset at {DATA_PATH}'
print('Using dataset:', DATA_PATH)

Using dataset: c:\Users\Admin\Desktop\infospace\data\claims.jsonl


In [ ]:
# claims.jsonl rows include keys like: id, input, output.claims, meta
raw = load_dataset('json', data_files=str(DATA_PATH), split='train')
print(raw)
print(raw[0].keys())
print(raw[0]['input'][:200])
print(raw[0]['output'].keys())

Generating train split: 5835 examples [00:00, 41887.69 examples/s]

Dataset({
    features: ['id', 'input', 'output', 'meta'],
    num_rows: 5835
})
dict_keys(['id', 'input', 'output', 'meta'])
Українська Центральна Рада (УЦР) — це перший український парламент, створений 17 березня 1917 року. Вона виникла в умовах революційних змін в Російській імперії та стала основою для проголошення незал
dict_keys(['claims'])


In [ ]:
SYSTEM_PROMPT = (
    'You extract factual claims from the given text. '
    'Return ONLY valid JSON with exactly two keys: input, claims. '
    'The claims key must be an array of concise claim strings. '
    'Do not add any extra keys or commentary.'
)

def to_prompt_completion(example):
    claims = example['output']['claims'] if example.get('output') else []
    target = {
        'input': example['input'],
        'claims': claims
    }
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['input']}
        ],
        'completion': [
            {'role': 'assistant', 'content': json.dumps(target, ensure_ascii=False)}
        ]
    }

dataset = raw.map(to_prompt_completion, remove_columns=raw.column_names)
dataset = dataset.train_test_split(test_size=0.05, seed=SEED)
train_ds = dataset['train']
eval_ds = dataset['test']

print(train_ds)
print(eval_ds)
print(train_ds[0])

Map: 100%|██████████| 5835/5835 [00:01<00:00, 3167.88 examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 5543
})
Dataset({
    features: ['prompt', 'completion'],
    num_rows: 292
})
{'prompt': [{'content': 'You extract factual claims from the given text. Return ONLY valid JSON with exactly two keys: input, claims. The claims key must be an array of concise claim strings. Do not add any extra keys or commentary.', 'role': 'system'}, {'content': 'Події Євромайдану призвели до значного погіршення відносин України з Росією, що виявилося в анексії Криму та початку війни на Донбасі. Це спричинило зміщення України в бік європейської інтеграції та посилення військової співпраці з НАТО.', 'role': 'user'}], 'completion': [{'content': '{"input": "Події Євромайдану призвели до значного погіршення відносин України з Росією, що виявилося в анексії Криму та початку війни на Донбасі. Це спричинило зміщення України в бік європейської інтеграції та посилення військової співпраці з НАТО.", "claims": ["Події Євромайдану призвели до значного по

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = str(ROOT / 'artifacts' / 'qwen2p5_0p5b_claims_lora')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_4bit = torch.cuda.is_available()
quant_config = None
if use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quant_config,
    torch_dtype=(torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'auto')
)

print('Model + tokenizer loaded.')

c:\Users\Admin\Desktop\infospace\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2227.98it/s]


Model + tokenizer loaded.


In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_length=1024,
    completion_only_loss=True,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and (not torch.cuda.is_bf16_supported()),
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config
)

print('Trainer ready.')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Truncating eval dataset: 100%|██████████| 292/292 [00:00<00:00, 2361.21 examples/s]


Trainer ready.


In [ ]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

print(train_result)
print('Saved adapter/model artifacts to:', OUTPUT_DIR)

NameError: name 'trainer' is not defined

## Inference helper (strict JSON output)

In [ ]:
def parse_json_object(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fallback: try to parse first JSON object span.
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end+1])
        raise

def generate_claims(text, max_new_tokens=384):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': text}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    gen_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    obj = parse_json_object(raw)

    # Enforce schema
    if not isinstance(obj, dict):
        raise ValueError('Model output is not a JSON object')
    if 'input' not in obj or 'claims' not in obj:
        raise ValueError(f'Missing required keys in output: {obj.keys()}')
    if not isinstance(obj['claims'], list):
        raise ValueError('claims must be a list')

    return obj, raw

In [ ]:
sample_text = eval_ds[0]['prompt'][1]['content']
result_obj, raw_output = generate_claims(sample_text)
print('Raw model output:
', raw_output)
print('
Parsed object:
', json.dumps(result_obj, ensure_ascii=False, indent=2))

In [ ]:
# Lightweight exact-match style evaluation on first N examples.
def normalize_claims(claims):
    return {c.strip().lower() for c in claims if isinstance(c, str)}

def evaluate_subset(ds, n=50):
    n = min(n, len(ds))
    overlaps = []
    valid_json = 0

    for i in range(n):
        inp = ds[i]['prompt'][1]['content']
        gold = parse_json_object(ds[i]['completion'][0]['content'])
        gold_set = normalize_claims(gold['claims'])

        try:
            pred, _ = generate_claims(inp)
            pred_set = normalize_claims(pred.get('claims', []))
            inter = len(gold_set & pred_set)
            union = len(gold_set | pred_set)
            overlaps.append(inter / union if union else 1.0)
            valid_json += 1
        except Exception:
            overlaps.append(0.0)

    return {
        'n': n,
        'valid_json_rate': valid_json / n,
        'mean_jaccard_claims': sum(overlaps) / n
    }

metrics = evaluate_subset(eval_ds, n=30)
print(metrics)